## Exercise 3: Misspecification testing

An introduction to the LM Test for Autocorrelation (aka. Breusch-Godfrey)!

This exercise class covers **questions 4, 5 and 6 of Homework 1**:

* Q4: code the Breusch-Godfrey LM test and the small sample corrected Edgerton-Shukur test.
* Q5: test for multivariate autocorrelation, normality and multivariate ARCH, and check against univariate built-in tests.
* Q6: code the companion matrix and a stability test based on its eigenvalues.

Consider the following VAR(p)-process:
\begin{equation*}
y_{t} = \nu + \rho_{1}y_{t-1} + \rho_{2}y_{t-2} + \cdots + \rho_{p}y_{t-p} + u_{t}
\end{equation*}
where $ \rho_{1}, \rho_{2}, \ldots, \rho_{p} $ are the autoregressive coefficients of the model, $ u_{t} $ is an error term and $ \nu $ a constant. Then, a test for autocorrelation in the innovations, $ u_{t} $, proposed by Breusch (1978) and Godfrey (1978) can be seen using the model:
\begin{equation*}
u_{t} = D_{1}u_{t-1} + D_{2}u_{t-2} + \cdots + D_{h}u_{t-h} + e_{t}
\end{equation*}
where $ e_{t} $ is a white noise error term. The test for no autocorrelation in $ u_{t} $ is then:
\begin{equation*}
\begin{aligned}
H_{0} &: D_{1} = \cdots = D_{h} = 0 \quad \text{versus} \\
H_{1} &: D_{i} \neq 0 \text{ for at least one } i \in \{1, \dots, h\}
\end{aligned}
\end{equation*}

In theory, $ u_{t} $ should be independent and identically distributed (iid.). However, the estimated residuals, $ \hat{u}_{t} $, that we need to use are not necessarily iid. The original variable $ y_{t-1} $ might be correlated with $ \hat{u}_{t-1} $, $ y_{t-2} $ with $ \hat{u}_{t-2} $ and so on. Therefore, we need to expand the model for $ \hat{u}_{t} $ to also include the original variables:
\begin{equation*}
\hat{u}_{t} = \nu + \rho_{1}y_{t-1} + \rho_{2}y_{t-2} + \cdots + \rho_{p}y_{t-p} + D_{1}\hat{u}_{t-1} + D_{2}\hat{u}_{t-2} + \cdots + D_{h}\hat{u}_{t-h} + e_{t}
\end{equation*}
where $ e_{t} $ is an auxiliary error term. At last, the Lagrange multiplier test statistic (also known as the Breusch-Godfrey test statistic) can be computed as:
\begin{equation*}
Q_{LM} = T \left(K - \text{tr}(\widetilde{\Sigma}_{u}^{-1} \widetilde{\Sigma}_{e})\right)
\end{equation*}
where $ \widetilde{\Sigma}_{u} = T^{-1} \sum_{t=1}^{T} \hat{u}_{t} \hat{u}_{t}^{\prime} $, $ \widetilde{\Sigma}_{e} = T^{-1} \sum_{t=1}^{T} \hat{e}_{t} \hat{e}_{t}^{\prime} $ and $ K $ is the number of equations (and variables in the system).

Note on $T$: throughout this notebook $T$ is the **effective** sample, i.e. the number of rows actually used in the estimation. With $p$ lags that is `y.shape[0] - p`.

As usual, we generate a VAR(2) model so we know the "data generating process":

In [ ]:
%reset -f

import numpy as np
import pandas as pd
from scipy.stats import chi2
from scipy.stats import f
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.diagnostic import het_arch
from statsmodels.stats.stattools import jarque_bera

np.random.seed(1)

# Parameters
N = 100  # Number of observations
rho1 = np.array([[0.8, 0],  # Autoregressive coefficient for lag 1
                 [0, 0.8]])

rho2 = np.array([[0.15, 0],  # Autoregressive coefficient for lag 2
                 [0, 0.15]])

alpha = np.array([0, 0])  # Mean of the VAR process

sigma = np.array([[1, 0],  # Covariance matrix of white noise
                  [0, 1]])

# Make the y-matrix ready to insert values
y = np.zeros((2, N))

# Generate white noise
epsilon = np.random.multivariate_normal([0, 0], sigma, N).T

# Generate the VAR(2) process with time trend
# Notice the time trend is added in the end of the equation below.
for t in range(2, N):
    y[:, t] = alpha + rho1 @ y[:, t-1] + rho2 @ y[:, t-2] + epsilon[:, t] + 0.1*t

y = y.T


1. Find and import the code for "VARlsExog()" from the student masterfile (s_master.py)!

The function will estimate your desired VAR-model using OLS. Read through it and make sure you understand the input and outputs to both functions "VARlsExog()" and "lagmatrix()".

2. Use the function to estimate a VAR(1) model without a constant, a trend and exogenous variables.

In [ ]:
from s_master import VARlsExog

p = 1
con = 0
tr = 0
exog = 0

Beta, CovBeta, tratioBeta, residuals_u, X, SIGMA_u = VARlsExog(y, p, con, tr, exog)

Let's take a look at the "structure" of Beta.

With 1 lag and no deterministic terms, Beta will have a 2x2 structure due to the way that Michael has constructed the VARlsExog()-code. It is easier to interpret Beta.T.

In [ ]:
print(f"{Beta.T}")

print(f"\n The {Beta.T[0,0]} is the effect of variable 1 at t-1 on variable 1 at time t.")
print(f"\n The {Beta.T[0,1]} is the effect of variable 2 at t-1 on variable 1 at time t.")

The first value of the FIRST row corresponds to the effect of variable 1 at t-1.
The second value of the FIRST row is the effect of the 2nd variable at t-1.

The columns of `Beta.T` are ordered the same way `VARlsExog` builds the regressor matrix:
first all the lags, then the constant (if con=1), then the trend (if tr=1), then any exogenous variables.

3. Now we need to check that the model is stable! We need to write our own functions for the companion matrix and then retrieve the eigenvalues to check the stability. **This is question 6 in the homework**, so no solution is attached to this exercise hehe.

* Construct the Companion Matrix. Remember its dimensions: it is $Kp \times Kp$. Take a look at this link to recall what it looks like:

URL to VAR-basics with companion-stuff: http://econweb.umd.edu/~chao/Teaching/Econ624/Econ624_Lecture_on_VAR.pdf

Two hints:

* `Beta.T` has the deterministic terms in the last columns, so slice out the first $Kp$ columns before you put them in the top block row of $A$.
* Sort the eigenvalue moduli in **descending** order. Then `e[0]` is the largest one, which is the one that decides stability. (If you sort ascending, the check below silently gives you the wrong answer.)

In [ ]:
def comp(Beta, p):
    # Insert your code here to construct the companion matrix
    return A

A = comp(Beta, p)


# Then we need to write a function to retrieve the eigenvalues and check the
# stability using the companion matrix.

def stabVAR(A):
    # Use numpy's eigenvalue function to retrieve the eigenvalues of A.
    # Remember that the VAR is stable if all eigenvalues are less than 1 in absolute value.
    return e

e = stabVAR(A)

print(f"The sorted eigenvalues are: {e}")

if e[0] < 1:
    print("The VAR is stable")
else:
    print("The VAR is not stable")

The sorted eigenvalues should be: [1.0192662141023392, 0.936352213278791]

So the VAR(1) without deterministic terms is **not** stable. That is what we expect: the data were generated with a time trend, and we have not given the model anything to pick it up with.

4. Reestimate the model including a trend (and still without a constant) and check if the model is now stable.
* This means that p is still 1, but tr = 1.

Check the structure of Beta now. When you transpose Beta, the third column is the effect of the linear time trend on the two equations.

In [ ]:
# FILL IN HERE: reestimate with tr = 1 and check stability again.

p = 1
con = 0
tr = 1
exog = 0

Beta, CovBeta, tratioBeta, residuals_u, X, SIGMA_u = # Fill in

print(f"{Beta.T}")

A = # Fill in
e = # Fill in
print(f"The sorted eigenvalues are: {e}")
if e[0] < 1:
    print("The VAR is stable")
else:
    print("The VAR is not stable")

5. Follow these steps to create a function that tests for residual MULTIVARIATE autocorrelation. In other words, we will construct the LM-test for autocorrelation proposed by Breusch and Godfrey, and also the small sample corrected test by Edgerton and Shukur. **This is question 4 in the homework.**

These tests are described in KL section 2.6.2. Useful references also include Lütkepohl (2004), Brüggeman, Lütkepohl and Saikkonen (2006) and Edgerton and Shukur (1999) for alternative descriptions of the test statistic. All three papers are in the homework folder.

$\quad$ $\quad$ (a) Construct lagged values for $u_{t-h}$ by using the function "lagmatrix(data, lags)" and choosing $h$

In [ ]:
# import the lagmatrix function from the masterfile
from s_master import lagmatrix

# We start with a single lag in the residuals
h = 1

# Use the lagmatrix-function here. Use residuals as input and specify the amount of lags h.
u_lags = # Fill in

$\quad$ $\quad$ (b) Regress the lagged residuals as exogenous variables on $y$.

This is the auxiliary regression! But note how it differs from the theoretical model I showed in the exercise class.

Note the `p` you pass here: the auxiliary regression keeps the **same $p$ lags of $y$** as the original model and adds $h$ lags of the residuals on top. So pass `p`, not `h`. (With `p = h = 1` you cannot tell the difference, but in question 11 we use `p = 3` and `h = 4` and then it matters.)

In [ ]:
Beta, CovBeta, tratioBeta, residuals_e, X, SIGMA_e = # Fill in

$\quad$ $\quad$ (c) Calculate the LM-test statistic using the formula:

\begin{equation*}
    Q_{LM} = T \left(K - \text{tr}(\widetilde{\Sigma}_{u}^{-1} \widetilde{\Sigma}_{e})\right)
\end{equation*}

$\quad$ $\quad$ where $T$ is the number of observations used for the estimation, and $\text{tr}$ is the trace of a matrix.

$\quad$ $\quad$ This is relevant for the homework! This is the Lütkepohl-version, where it is simpler to compute with an asymptotic $\chi^2$ distribution.

$\quad$ $\quad$ Careful with $T$: `y.shape[0]` is 100, but only $100 - p$ rows are actually used.

In [ ]:
t, K = y.shape

# We call it LML because it is the LM-test, but the Lütkepohl version!

LML = # Fill in

$\quad$ $\quad$ (d) At last, calculate the asymptotic $\chi^2(hK^2)$ distribution and use this to find the p-value:

In [ ]:
LMLpval = # Fill in

print(round(LML, 2), round(LMLpval, 4))

If done correctly, the result should be (with p = 1, tr = 1 and con = 0):

[LML, LMLpval] = [23.29, 0.0001]

Does the model suffer from residual autocorrelation? Extend the model by including another lag (p = 2) and increase the horizon by 1 (h = 2). Does the model still suffer from residual autocorrelation?

In [ ]:
# Now we set p = 2 and h = 2

p = 2
con = 0
tr = 1
exog = 0
h = 2

# Fill in: reestimate, build u_lags, run the auxiliary regression and
# recompute LML and LMLpval.

If done correctly, the result should be:

[LML, LMLpval] = [6.17, 0.6279]

6. Collect all the equations and insert them into one function called "VARLMtest(y, p, con, tr, exog, h)":

Note the extra return value `residuals_u` - we reuse those residuals for the ARCH and normality tests further down, so it saves us re-estimating.

In [ ]:
def VARLMtest(y, p, con, tr, exog, h):

    # Fill in
    # Fill in
    # Fill in
    # Fill in
    # Fill in
    # Fill in

    return LML, LMLpval, residuals_u

LML, LMLpval, _ = VARLMtest(y, p, con, tr, exog, h)

print(round(LML, 2), round(LMLpval, 4))

Edgerton and Shukur (1999) propose a small sample version of the LM test given as:

\begin{equation*}
\text{FLMh}=\left(\left(\det(\Sigma_{u})\det(\Sigma_{e})^{-1}\right)^{\frac{1}{s}}-1\right)\cdot\left(\frac{Ns-q}{Km}\right)
\end{equation*}

where:

\begin{equation*}
\begin{aligned}
m & =Kh\\
q & =\frac{1}{2}Km-1\\
s & =\left(\frac{K^{2}\cdot m^{2}-4}{K^{2}+m^{2}-5}\right)^{\frac{1}{2}}\\
N & =T-Kp-m-\frac{1}{2}(K-m+1)
\end{aligned}
\end{equation*}

FLMh meaning: F stands for F-distribution, LM is Lagrange multiplier as you know, and h means that it is multivariate for h lags.

The statistic is $F(Km,\; Ns-q)$ distributed. Use the same effective $T$ here as you used in $Q_{LM}$, otherwise the two tests are not computed on the same sample.

7. Include this in the "VARLMtest()". So basically just write up the FLMh equation :)

In [ ]:
p = 2
con = 0
tr = 1
exog = 0
h = 2

def VARLMtest(y, p, con, tr, exog, h):

    # Fill in from the previous function

    m = # Fill in
    q = # Fill in
    s = # Fill in
    N = # Fill in

    FLMh = # Fill in
    FLMh_pval = # Fill in


    Results = [[LML, FLMh], [LMLpval, FLMh_pval], [h, h]]

    lm_table = pd.DataFrame({
        'Measure': pd.Categorical(['Test statistic', 'p-value', 'Lag order']),
        'Breusch_Godfrey': [row[0] for row in Results],
        'Edgerton_Shukur': [row[1] for row in Results]
    })

    return Results, lm_table, residuals_u

Results, lm_table, _ = VARLMtest(y, p, con, tr, exog, h)

lm_table

If done correctly and setting p = 2 and h = 2, one should get:

| Measure         | Breusch-Godfrey | Edgerton-Shukur |
|-----------------|-----------------|-----------------|
| Test statistic  | 6.17            | 0.73            |
| p-value         | 0.63            | 0.66            |
| Lag order       | 2               | 2               |

8. Test for univariate autocorrelation.

This is the "check these results" part of question 5 in the homework: we compare our own multivariate test against the built-in univariate ones.

In [ ]:
p = 2
con = 0
tr = 1
exog = 0

t, K = y.shape

Beta, CovBeta, tratioBeta, residuals, X, SIGMA_u = VARlsExog(y, p, con, tr, exog)

lags = [1, 2, 3, 4, 5, 6] # Example, could be any number of lags

p_values_array = np.zeros((len(lags), K))

# Perform Ljung-Box test for each residual series and store p-values
for i in range(K):
    df = acorr_ljungbox(residuals[:, i], lags=lags, return_df=True)
    p_values_array[:, i] = df['lb_pvalue'].values  # Accessing the 'lb_pvalue' column as an array

# Convert the array into a DataFrame for better readability
p_values_df = pd.DataFrame(p_values_array, index=[f"P-Value Lag {lag}" for lag in lags],
                           columns=["Variable 1", "Variable 2"]) # Remember to change the column names if needed

p_values_df

Per Kilian and Lütkepohl (2017, p 67):

Given the auxiliary model of horizontally stacked residuals:

\begin{equation*}
\text{vech}(\hat{u}_t \hat{u}_t') = \delta_0 + D_1 \text{vech}(\hat{u}_{t-1} \hat{u}_{t-1}') + \dots + D_q \text{vech}(\hat{u}_{t-q} \hat{u}_{t-q}') + e_t.
\end{equation*}

We have that there are no ARCH effects given that the following null hypothesis is true:

\begin{equation*}
H_0 : D_1 = \cdots = D_q = 0 \quad \text{versus} \quad H_1 : D_i \neq 0 \, \text{for at least one} \, i \in \{1, \dots, q\}.
\end{equation*}

The test for residual ARCH effects can be calculated using the LM-test:

\begin{equation*}
LM_{\text{ARCH}}(q) = \frac{1}{2} T K (K + 1) \left( 1 - \frac{2}{K (K + 1)} \text{tr}(\hat{\Omega} \hat{\Omega}_{0}^{-1}) \right)
\end{equation*}

where $\hat{\Omega}$ is the residual covariance matrix of the auxiliary regression, $\hat{\Omega}_{0}$ is the covariance matrix under the null hypothesis of no ARCH, and $T$ is the sample size. The null hypothesis is that there is no ARCH effect up to lag $q$.

Note the degrees of freedom: $qK^2(K+1)^2/4$. That grows very fast in $K$. With $K = 5$ and $q = 2$ it is already 450, so do not be surprised by a large test statistic in question 11.

9. Now we need to test for multivariate heteroskedasticity.
Use the "march()"-function from the s_master file.

Look it up to see what you need as input.

In [ ]:
from s_master import march

lag = 2

test, march_table = # Fill in

march_table

... and univariate

In [ ]:
hypothesis_results = np.zeros(K)
p_values_results = np.zeros(K)
stat_results = np.zeros(K)
crit_value_results = np.zeros(K)

# Perform archtest for each variable using a loop
for i in range(K):
    stat, pValue, fval, _ = het_arch(residuals[:, i], nlags=lag)
    hypothesis_results[i] = 1 if pValue < 0.05 else 0  # Hypothesis result based on p-value
    p_values_results[i] = pValue
    stat_results[i] = stat
    crit_value_results[i] = chi2.ppf(1 - 0.05, lag)

row_names = ['Variable 1', 'Variable 2']  # Adjust names as necessary

# Creating a table with the results
het_table = pd.DataFrame({
    'Hypothesis': hypothesis_results,
    'P-Value': p_values_results,
    'Test statistic': stat_results,
    'CriticalValue': crit_value_results
}, index=row_names)

het_table

As per Kilian and Lütkepohl (2017, p 66):

A test for skewness is given by:

\begin{equation*}
\lambda_{3} = T\,\frac{\hat{b}_{3}^{\prime} \hat{b}_{3}}{6} \overset{d}{\rightarrow} \chi^{2}(K)
\end{equation*}

where $\hat{b}_{j}^{\prime} = (\hat{b}_{1j}, \dots, \hat{b}_{Kj})^{\prime}$ and $\hat{b}_{kj} = \frac{1}{T} \sum_{t=1}^{T} (\hat{u}_{kt}^{s})^{j}$. $K$ is the number of variables, while the residuals are standardized such that $\hat{u}_{t}^{s} = P^{-1} \hat{u}_{t}$ and $\tilde{\Sigma}_{u} = P P^{\prime}$. Doornik and Hansen (1994) use the square root matrix of $\tilde{\Sigma}_{u}$, whereas Lütkepohl (2005, Chapter 4) uses the Cholesky decomposition.

A test for excess kurtosis is given by:

\begin{equation*}
\lambda_{4} = T\,\frac{(\hat{b}_{4} - 3_{K})^{\prime} (\hat{b}_{4} - 3_{K})}{24} \overset{d}{\rightarrow} \chi^{2}(K)
\end{equation*}

where $3_{K} = (3, \dots, 3)^{\prime}$ is a $K \times 1$ vector.

At last, a test for non-normality is given by:

\begin{equation*}
\lambda = \lambda_{3} + \lambda_{4} \sim \chi^{2}(2K)
\end{equation*}

Note the factor $T$ and the constant 6 in $\lambda_3$: that is what `multnorm()` actually computes
(`n * b.T @ b / 6`). Check it against the code if in doubt.

This is why `multnorm()` reports two columns: the two authors disagree about how to whiten the residuals, so you get two numbers for the same null hypothesis.

10. Now we perform a test for multivariate normality (use "multnorm()" from s_master.py)

In [ ]:
from s_master import multnorm

norm, multnorm_table = # Fill in

multnorm_table

... and univariate

In [ ]:
hypothesis_results = np.zeros(K)
p_values_results = np.zeros(K)
stat_results = np.zeros(K)
crit_value_results = np.zeros(K)

# Perform jarque_bera test for each variable using a loop
for i in range(K):
    (JB, JBpv, skew, kurtosis) = jarque_bera(residuals[:, i])
    hypothesis_results[i] = 1 if JBpv < 0.05 else 0  # Hypothesis result based on p-value
    p_values_results[i] = JBpv
    stat_results[i] = JB
    crit_value_results[i] = chi2.ppf(1 - 0.05, 2)  # Degrees of freedom for JB test is 2

row_names = ['Variable 1', 'Variable 2']  # Adjust names as necessary

# Creating a table with the results
jb_table = pd.DataFrame({
    'Hypothesis': hypothesis_results,
    'P-Value': p_values_results,
    'Test statistic': stat_results,
    'CriticalValue': crit_value_results
}, index=row_names)

jb_table

Note 1: The critical value only approaches $\chi^2(2)$ in large samples, hence the above Critical Value is not correct in small samples.

Note 2: Asymptotic properties of the univariate tests can be found on Absalon.

11. Now apply everything to the dataset from Bjørnland (2009), i.e. to the homework.

The specification from the homework hints is:

* $x_t = [\;rfor,\; lgdp,\; \pi,\; dlrexc,\; rdom\;]$, so $K = 5$
* lag order $p = 3$ (this is what you found with the information criteria and the top-down sequence in Exercise 2)
* a constant and a linear trend
* three exogenous dummies: du93Q1, du92Q3, du95Q4
* sample 1983:1 - 2004:4, so $T = 88$ raw observations

If you saved the data from Exercise 1 with

    result.to_csv('result.csv', index=False)

you can load it in with the following. A copy of `result.csv` is in this folder if you do not have your own.

In [ ]:
# Load the transformed data from Exercise 1
y_df = pd.read_csv('result.csv', index_col=0)

print(y_df.columns.tolist())
print(y_df.shape)

# The five endogenous variables
y = y_df[['rfor', 'lgdp', 'pi', 'dlrexc', 'rdom']].to_numpy()

# The three dummies are EXOGENOUS, not endogenous
dummies_full = y_df[['du93Q1', 'du92Q3', 'du95Q4']].to_numpy()

T, K = y.shape
print(f"T = {T}, K = {K}")

Careful with the dummies: `VARlsExog` stacks `exog` next to the lagged regressors, and those only have $T-p$ rows. So you have to drop the first $p$ rows of the dummy matrix before you pass it in.

And one more thing the exercise above does not show you: when the original model has exogenous dummies, **the auxiliary regression must keep them too**. Otherwise you are not testing the same model. So inside `VARLMtest` the auxiliary call needs `np.hstack([exog, u_lags])` rather than just `u_lags`.

In [ ]:
p = 3
con = 1
tr = 1
h = 4        # quarterly data, so 4 is a natural starting point
lag = 2

exog = dummies_full[p:, :]   # drop the first p rows

# Fill in: run VARLMtest on this specification.
# Remember to extend it so the auxiliary regression also keeps the dummies.

Results, lm_table, residuals = # Fill in

lm_table

Test for multivariate ARCH

In [ ]:
test, march_table = # Fill in

march_table

Multivariate normality test

In [ ]:
norm, multnorm_table = # Fill in

multnorm_table

Finally, check the stability of the selected model using your own `comp()` and `stabVAR()` from question 3.

In [ ]:
Beta, CovBeta, tratioBeta, residuals, X, SIGMA_u = VARlsExog(y, p, con, tr, exog)

A = # Fill in
e = # Fill in

print(f"The largest eigenvalue modulus is: {e[0]:.4f}")
if e[0] < 1:
    print("The VAR is stable")
else:
    print("The VAR is not stable")

Questions to discuss:

* Do the Breusch-Godfrey and the Edgerton-Shukur tests agree? If not, which one do you trust with $T = 85$ and 20 regressors per equation, and why?
* Does the lag order you chose in Exercise 2 survive the misspecification tests? Try $p = 1, 2, 3, 4, 5$ and look at the Edgerton-Shukur p-value.
* We reject normality. How much should that worry us? What exactly did we need normality for?
* Summarize all of this in one table - that is literally what question 5 of the homework asks for.